# Davinci — 500M Decoder Architecture & Compute Planning

This notebook defines an auditable decoder-only Transformer configuration, computes parameter counts from tensor dimensions, estimates attention/KV-cache memory, and exports a machine-readable model config.

It is a **design and planning notebook**, not a training run. The default configuration is a candidate near 500M parameters; it must be validated against the selected tokenizer, framework implementation, hardware, and held-out evaluations before training.

## 1. Install and set reproducibility

In [ ]:
!pip -q install -U pandas

In [ ]:
import json, math
from pathlib import Path
import pandas as pd

SEED=42
OUT=Path("/content/davinci_architecture")
OUT.mkdir(parents=True,exist_ok=True)

## 2. Candidate architecture

Default candidate: 28 decoder layers, hidden size 1280, 20 query heads, 4 key/value heads, head dimension 64, SwiGLU intermediate size 3328, 32K vocabulary, and tied input/output embeddings.

The model uses grouped-query attention (GQA). A hybrid local/global attention schedule is specified as a design choice, not a proven optimum. For example, use a full-context layer every fourth layer and sliding-window attention in the others. Check that the chosen training/inference framework implements the same semantics.

In [ ]:
CONFIG = {
    "model_type": "davinci_decoder",
    "architectures": ["DavinciForCausalLM"],
    "vocab_size": 32000,             # update after tokenizer is frozen
    "hidden_size": 1280,
    "num_hidden_layers": 28,
    "num_attention_heads": 20,
    "num_key_value_heads": 4,
    "head_dim": 64,
    "intermediate_size": 3328,
    "hidden_act": "swiglu",
    "norm_type": "rmsnorm",
    "norm_eps": 1e-5,
    "position_encoding": "rope",
    "rope_theta": 10000.0,
    "max_position_embeddings": 4096, # initial pretraining context, not 500K
    "attention_bias": False,
    "mlp_bias": False,
    "tie_word_embeddings": True,
    "attention_schedule": {
        "type": "periodic_global_plus_sliding_window",
        "global_every_n_layers": 4,
        "local_window": 4096,
        "global_layers_are_zero_indexed": True
    },
    "dropout": 0.0,
    "initializer_range": 0.02,
    "parameter_dtype": "bfloat16"
}
print(json.dumps(CONFIG,indent=2))

## 3. Validate dimensions and calculate parameter count

The calculation assumes standard dense Q/K/V/O projections, SwiGLU with three linear matrices, RMSNorm scales, no linear biases, and tied token embedding/output weights. It excludes small implementation-specific buffers and any extra modules not represented here.

In [ ]:
def validate_config(c):
    assert c["hidden_size"] == c["num_attention_heads"] * c["head_dim"]
    assert c["num_attention_heads"] % c["num_key_value_heads"] == 0
    assert c["intermediate_size"] > 0 and c["num_hidden_layers"] > 0
    assert c["vocab_size"] > 0 and c["max_position_embeddings"] > 0
    assert c["attention_schedule"]["global_every_n_layers"] >= 1
    assert c["attention_schedule"]["local_window"] > 0

def parameter_breakdown(c):
    d=c["hidden_size"]; h=c["num_attention_heads"]; kv=c["num_key_value_heads"]
    hd=c["head_dim"]; f=c["intermediate_size"]; L=c["num_hidden_layers"]; V=c["vocab_size"]
    q=d*d
    k=kv*hd*d
    v=kv*hd*d
    o=d*d
    attention=q+k+v+o
    # SwiGLU: gate, up, and down projections
    mlp=3*d*f
    norms=2*d
    per_layer=attention+mlp+norms
    embeddings=V*d
    final_norm=d
    lm_head=0 if c["tie_word_embeddings"] else V*d
    total=L*per_layer+embeddings+final_norm+lm_head
    return {"q_projection":q,"k_projection":k,"v_projection":v,"o_projection":o,
            "attention_per_layer":attention,"swiglu_mlp_per_layer":mlp,
            "two_rmsnorms_per_layer":norms,"all_decoder_layers":L*per_layer,
            "token_embedding":embeddings,"final_norm":final_norm,
            "untied_lm_head":lm_head,"total_parameters":total}
validate_config(CONFIG)
breakdown=parameter_breakdown(CONFIG)
display(pd.DataFrame([{"component":k,"parameters":v} for k,v in breakdown.items()]))
print(f"Total: {breakdown['total_parameters']/1e6:.2f} million parameters")

## 4. Compare nearby candidates

This table helps show how depth, width, and vocabulary affect parameter count. Keep head dimension and GQA ratio valid. Do not assume a 500M label means the model will fit a particular GPU; activation and optimizer memory also matter.

In [ ]:
def candidate(name,layers,hidden,heads,kv_heads,head_dim,ffn,vocab=CONFIG["vocab_size"]):
    c={**CONFIG,"num_hidden_layers":layers,"hidden_size":hidden,
       "num_attention_heads":heads,"num_key_value_heads":kv_heads,
       "head_dim":head_dim,"intermediate_size":ffn,"vocab_size":vocab}
    validate_config(c)
    return {"candidate":name,"layers":layers,"hidden":hidden,"heads":heads,
            "kv_heads":kv_heads,"ffn":ffn,"vocab":vocab,
            "parameters":parameter_breakdown(c)["total_parameters"]}
candidates=[
    candidate("shallower",24,1280,20,4,64,3328),
    candidate("default",28,1280,20,4,64,3328),
    candidate("deeper",30,1280,20,4,64,3328),
    candidate("wider",24,1408,22,2,64,3584),
]
display(pd.DataFrame(candidates).assign(parameters_M=lambda x:(x.parameters/1e6).round(2)))

## 5. KV-cache memory estimator

For standard autoregressive inference, approximate KV cache bytes as:
2 × layers × batch × sequence length × KV heads × head dimension × bytes per element.

This excludes weights, activations, allocator fragmentation, temporary attention buffers, and framework overhead. It assumes all layers cache keys/values for the full sequence; sliding-window layers may use less if the runtime actually evicts old cache entries.

In [ ]:
DTYPE_BYTES={"fp32":4,"fp16":2,"bf16":2,"int8":1}
def kv_cache_bytes(c,batch,seq,dtype="bf16",local_window=None):
    bytes_per=DTYPE_BYTES[dtype]
    layers=c["num_hidden_layers"]; kv=c["num_key_value_heads"]; hd=c["head_dim"]
    # Per-layer token count. Full/global layers retain full sequence; local layers cap at window.
    every=c["attention_schedule"]["global_every_n_layers"]
    window=local_window or c["attention_schedule"]["local_window"]
    total_tokens=0
    for layer in range(layers):
        is_global=((layer+1)%every==0)
        total_tokens += seq if is_global else min(seq,window)
    return 2*batch*kv*hd*total_tokens*bytes_per

rows=[]
for seq in [4096,8192,32768,65536,131072,500000]:
    for batch in [1,4]:
        n=kv_cache_bytes(CONFIG,batch,seq,"bf16")
        rows.append({"sequence":seq,"batch":batch,"KV_cache_GiB":n/(1024**3)})
kv_df=pd.DataFrame(rows)
display(kv_df.pivot(index="sequence",columns="batch",values="KV_cache_GiB").round(3))

## 6. Attention compute and 500K context warning

Dense full-context attention has quadratic sequence-length compute. Sliding-window attention reduces local-layer cost only if the kernel and runtime truly support it. Periodic global layers still introduce long-range interactions but can become expensive at very long lengths.

A 500,000-token context is **not** a sensible default training target for this 500M model. Treat it as a research goal requiring specialized attention kernels, memory planning, staged training, and retrieval/long-context evaluation. Start at 4K or 8K, then increase only after measured stability and quality.

In [ ]:
def attention_pairwise_score_count(c,seq):
    L=c["num_hidden_layers"]; every=c["attention_schedule"]["global_every_n_layers"]
    window=c["attention_schedule"]["local_window"]
    total=0
    for layer in range(L):
        global_layer=((layer+1)%every==0)
        total += seq*seq if global_layer else seq*min(seq,window)
    return total*c["num_attention_heads"]

attn_rows=[]
for seq in [4096,8192,32768,65536,131072,500000]:
    attn_rows.append({"sequence":seq,"attention_query_key_pairs":attention_pairwise_score_count(CONFIG,seq)})
display(pd.DataFrame(attn_rows))

## 7. Rough training memory budget

Parameter count alone is not enough. This rough lower-bound estimator shows weight storage only. Adam optimizer states, gradients, activations, temporary buffers, and distributed-training overhead can multiply memory needs substantially. Use measured peak memory from a small benchmark before committing to a hardware plan.

In [ ]:
total_params=breakdown["total_parameters"]
memory_rows=[]
for dtype,bytes_per in DTYPE_BYTES.items():
    memory_rows.append({"dtype":dtype,"weight_only_GiB":total_params*bytes_per/(1024**3)})
display(pd.DataFrame(memory_rows).round(3))
print("Illustrative Adam training often needs several times the weight-only memory; actual usage depends on optimizer, precision, sharding, and activation checkpointing.")

## 8. Export a framework-neutral config and report

In [ ]:
CONFIG["parameter_count_estimate"]=breakdown["total_parameters"]
CONFIG["parameter_count_method"]="Dense GQA attention + SwiGLU + RMSNorm; tied embeddings; biases disabled"
CONFIG["design_status"]="candidate_not_benchmarked"
CONFIG["context_plan"]={"initial_context":4096,"later_targets":[8192,16384,32768,65536,131072],"500k_status":"research_goal_not_default"}
config_path=OUT/"davinci_500m_config.json"
report_path=OUT/"davinci_architecture_report.json"
config_path.write_text(json.dumps(CONFIG,ensure_ascii=False,indent=2),encoding="utf-8")
report={"parameter_breakdown":breakdown,"candidate_comparison":candidates,
        "kv_cache_bf16_batch1":{str(s):kv_cache_bytes(CONFIG,1,s,"bf16") for s in [4096,8192,32768,65536,131072,500000]},
        "assumptions":["Tied embeddings","No projection biases","SwiGLU three-matrix MLP",
                       "KV cache estimate assumes runtime supports local-window eviction",
                       "Does not include optimizer/activation/temporary memory"]}
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
print(config_path,report_path)

## 9. Download configuration artifacts

In [ ]:
from google.colab import files
files.download(str(config_path))
files.download(str(report_path))

## 10. Pre-training architecture sign-off checklist

- Freeze the tokenizer and verify its actual vocabulary size before setting `vocab_size`.
- Confirm model parameter count against the instantiated framework model, including tied-weight behavior.
- Confirm RoPE implementation and scaling policy; do not extrapolate to 500K by changing a single scalar.
- Verify GQA cache layout and sliding-window semantics in the chosen attention kernel.
- Run a forward/backward smoke test at short context; check loss decrease and gradient stability.
- Benchmark peak allocated/reserved GPU memory at target microbatch and sequence lengths.
- Record throughput, tokens/sec, GPU utilization, and checkpoint size.
- Compare local/global attention schedules on held-out long-context retrieval and multi-hop tasks.
- Keep the 500K goal separate from the initial supported context until the model passes length-specific evaluations.

**No architecture is certified as optimal by this notebook.** The exported JSON is a reproducible starting specification; actual implementation parity and downstream performance must be tested.